# Surrogate Factory — UCAirfoils
## Chapter 10. Model Storage
Objectives:
- Store the trained models and the fitted preprocessor.
- Store the deployable sklearn pipelines plus a self-describing `manifest.json`.
- Run an integration test that reloads the **stored** artifacts and predicts,
  so a store that cannot be consumed downstream fails here.

Destination comes from `metadata/SF_10_Model_Storage.yaml` (`storage_path`).
A relative path resolves against `data.folder`.


### 0. Workflow initialisation

In [ ]:
from IPython.display import display, HTML, JSON
from surrogate_factory.workflow import Workflow

workflow = Workflow("pipeline_config.yaml")
workflow.resume()


### 10. Model Storage

In [ ]:
workflow.import_metadata(stage_name="SF_10_Model_Storage")

#### 10.1 Store models and preprocessor

In [ ]:
from storage.model import upload_model
stored_models = upload_model(workflow)


#### 10.2 Store deployable pipelines + manifest

In [ ]:
from storage.pipeline import save_pipeline
stored_pipelines = save_pipeline(workflow)


#### 10.3 Integration test
Reloads each pipeline from the store and predicts on a few test rows.

In [ ]:
from storage.integration import model_test

job = workflow.config["job_name"]
Test_set = workflow.load_data(job + "_Test_set.csv")

test_result = model_test(workflow, Test_set)
JSON(test_result)


#### 10.4 Store contents

In [ ]:
from pathlib import Path
import pandas as pd

store = Path(workflow.metadata.get_step_data(
    ["metadata", "Model_Storage", "Store", "storage_path"]) or "model_store")
if not store.is_absolute():
    store = Path(workflow.config["data.folder"]) / store

rows = [{"file": f.name, "size_KB": round(f.stat().st_size / 1024, 1)}
        for f in sorted(store.glob("*")) if f.is_file()]
print(f"{store}\n")
display(pd.DataFrame(rows))


### Save

In [ ]:
workflow.save_metadata()